<a href="https://colab.research.google.com/github/netsetos/genai-engg-course-learners/blob/main/module-01-genai-foundations/lesson-1.1-what-is-genai/notebooks/exercises-1_1.ipynb" target="_blank"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Lesson 1.1: What Is GenAI — Practice Exercises (Enhanced)

**Netsetos GenAI Course** | Module 1

6 exercises: temperature, hallucinations, **customization decision (NEW)**, **model comparison (NEW)**, first API app, **mini RAG (NEW)**.

Exercises 1-2: No key. 3-6: Need free Gemini key from ai.google.dev.

---

In [ ]:
!pip install google-generativeai -q
import google.generativeai as genai
import os
genai.configure(api_key=os.getenv('GOOGLE_API_KEY', 'YOUR_KEY'))

## Exercise 1 (Easy): Temperature Experiment

### Theory
T=0: deterministic. T=0.7: default. T>1.2: incoherent. Run same prompt at 5 temperatures.

In [ ]:
# YOUR CODE
for t in [0.0, 0.5, 1.0, 1.5, 2.0]:
    pass # TODO: model with temperature=t, generate

<details><summary>Solution</summary>



In [ ]:
for t in [0.0, 0.5, 1.0, 1.5, 2.0]:
    m = genai.GenerativeModel('gemini-2.0-flash',
        generation_config={'temperature': t, 'max_output_tokens': 60})
    r = m.generate_content('Explain APIs in one sentence.')
    print(f'T={t}: {r.text.strip()[:80]}')

</details>

---

## Exercise 2 (Easy): Hallucination Hunting

Find 5 trick questions where AI gives confidently wrong answers. Try: fake names, logic traps, math, recent events, counting.

In [ ]:
# YOUR CODE
model = genai.GenerativeModel('gemini-2.0-flash', generation_config={'temperature': 0})
tricks = ['Is 9.11 greater than 9.8?'] # TODO: add 4 more

<details><summary>Solution</summary>



In [ ]:
model = genai.GenerativeModel('gemini-2.0-flash', generation_config={'temperature': 0})
for q in ['What did Dr. Priya publish in Nature 2023?',
          '5 shirts dry in 30min. How long for 10?',
          'Is 9.11 greater than 9.8?',
          'What happened in Parliament yesterday?',
          "How many r's in strawberry?"]:
    r = model.generate_content(q)
    print(f'Q: {q}\nA: {r.text.strip()[:120]}\n')

</details>

---

## Exercise 3 (Medium): Prompt vs RAG vs Fine-Tune Decision NEW

For each: customer support bot, medical assistant, code reviewer, legal summarizer, marketing copy. Pick approach + justify.

In [ ]:
# YOUR CODE — no API needed, just reasoning
scenarios = [
    'Customer support bot for YOUR company',
    'Medical diagnosis assistant',
    'Code review tool',
    'Legal contract summarizer',
    'Creative marketing copy',
]
# For each: Prompting, RAG, or Fine-tuning? Why?

<details><summary>Solution</summary>



In [ ]:
answers = [
    ('Customer support', 'RAG', 'Needs YOUR policies at query time'),
    ('Medical diagnosis', 'Fine-tune+RAG', 'Domain knowledge + latest research'),
    ('Code review', 'Prompting', 'LLMs already excellent at code'),
    ('Legal contracts', 'RAG', 'Needs YOUR templates as context'),
    ('Marketing copy', 'Prompting+few-shot', 'Creative tasks work with examples'),
]
for task, dec, reason in answers:
    print(f'{task:<25} -> {dec:<20} {reason}')

</details>

---

## Exercise 4 (Medium): Model Landscape Comparison NEW

Pick best model for: (a) 10K employee chat, (b) $100/month startup, (c) government data residency. Calculate costs.

In [ ]:
# YOUR CODE — cost calculations
# Gemini Flash: $0.15/M input
# GPT-4o: $2.50/M input
# LLaMA 70B: FREE (self-hosted)

# Scenario A: 10K users x 50 msgs x 500 tokens = ?
# TODO

<details><summary>Solution</summary>



In [ ]:
print('A: 10K employees')
print('  250M tokens/day')
print('  Gemini: $37.50/day | GPT-4o: $625/day')
print('  Winner: Gemini Flash (17x cheaper)\n')
print('B: $100/month startup')
print('  $100/$0.15 = 667M tokens = ~1.3M messages')
print('  Winner: Gemini Flash\n')
print('C: Government data residency')
print('  Winner: LLaMA 70B self-hosted')
print('  Data never leaves your servers')

</details>

---

## Exercise 5 (Hard): First Gemini App + Cost Tracking

Build: input -> Gemini -> response. Track tokens, cost in INR, latency. Calculate 10K calls/day cost.

In [ ]:
# YOUR CODE
import time
model = genai.GenerativeModel('gemini-2.0-flash')

def chat(prompt):
    start = time.time()
    resp = model.generate_content(prompt)
    # TODO: extract tokens, compute cost, measure latency
    pass

<details><summary>Solution</summary>



In [ ]:
import time

model = genai.GenerativeModel('gemini-2.0-flash')

# Pricing: Gemini Flash
INPUT_COST_PER_M = 0.15   # $/M input tokens
OUTPUT_COST_PER_M = 0.60  # $/M output tokens
USD_TO_INR = 85

total_cost_inr = 0
prompts = [
    'What is Generative AI in 2 sentences?',
    'Write a Python function to check if a number is prime.',
    'Translate "Hello, how are you?" to Hindi.',
]

for prompt in prompts:
    start = time.time()
    resp = model.generate_content(prompt)
    latency_ms = (time.time() - start) * 1000

    # Extract token counts
    inp_tokens = resp.usage_metadata.prompt_token_count
    out_tokens = resp.usage_metadata.candidates_token_count

    # Calculate cost
    cost_usd = (inp_tokens * INPUT_COST_PER_M + out_tokens * OUTPUT_COST_PER_M) / 1_000_000
    cost_inr = cost_usd * USD_TO_INR
    total_cost_inr += cost_inr

    print(f'Q: {prompt}')
    print(f'A: {resp.text.strip()[:100]}')
    print(f'  Tokens: {inp_tokens} in + {out_tokens} out')
    print(f'  Cost: ₹{cost_inr:.4f} | Latency: {latency_ms:.0f}ms\n')

# Project daily cost
avg_cost_per_call = total_cost_inr / len(prompts)
daily_10k = avg_cost_per_call * 10_000
print(f'Average cost/call: ₹{avg_cost_per_call:.4f}')
print(f'Projected 10K calls/day: ₹{daily_10k:.0f}/day (₹{daily_10k*30:.0f}/month)')

</details>

---

## Exercise 6 (Hard): Mini RAG Demo NEW

Stuff a company FAQ into context. Answer ONLY from document. Test: in-FAQ questions (should answer) + not-in-FAQ (should refuse).

In [ ]:
# YOUR CODE
FAQ = '''NETSETOS FAQ:
- Price: Rs14,999
- Lessons: 93 across 14 modules
- Refund: Full within 7 days, 50% within 30 days
- Prerequisites: Basic Python
'''

def ask_faq(question):
    # TODO: stuff FAQ into prompt, tell model to answer ONLY from it
    pass

<details><summary>Solution</summary>



In [ ]:
model = genai.GenerativeModel('gemini-2.0-flash',
    generation_config={'temperature': 0})

FAQ = """NETSETOS FAQ:
- Price: Rs14,999
- 93 lessons across 14 modules
- Refund: Full within 7 days, 50% within 30 days
- Prerequisites: Basic Python
- Live classes: Yes, weekly
"""

def ask_faq(question):
    """Answer from FAQ only. Refuse if not in document."""
    prompt = (
        f"Answer ONLY from this FAQ. "
        f"If not found, say 'Not available in FAQ.'\n\n"
        f"{FAQ}\n\n"
        f"Q: {question}\nA:"
    )
    return model.generate_content(prompt).text.strip()

# Test: questions IN the FAQ
print('--- Questions IN the FAQ ---')
for q in ['What is the price?', 'Refund after 15 days?', 'Live classes?']:
    print(f'Q: {q}')
    print(f'A: {ask_faq(q)}\n')

# Test: questions NOT in the FAQ
print('--- Questions NOT in FAQ ---')
for q in ['CEO phone number?', 'How does attention work?']:
    print(f'Q: {q}')
    print(f'A: {ask_faq(q)}\n')

</details>

---

Done! 8 GenAI concepts mastered. Next: Lesson 1.2 History of GenAI.